# A13. 베이즈 정리 응용: 지문 문제 & 택시 문제

**핵심 주제**: 기저율 무시 오류 (Base Rate Fallacy)

---

## 학습 목표
1. 베이즈 정리를 실제 문제에 적용하는 능력
2. **기저율 무시 오류**가 왜 위험한지 이해
3. 직관과 수학적 결과의 괴리를 경험

## 1. 환경 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트 설정 (Malgun Gothic)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Seaborn 스타일
sns.set_theme(style='whitegrid', font='Malgun Gothic')

# 고해상도 출력
plt.rcParams['figure.dpi'] = 120

print('환경 설정 완료')

## 2. 베이즈 정리 복습

### 베이즈 정리 (Bayes' Theorem)

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

각 항의 의미:

| 항 | 이름 | 의미 |
|---|---|---|
| $P(A \mid B)$ | **사후 확률** (Posterior) | 증거 $B$를 관찰한 후 $A$의 확률 |
| $P(B \mid A)$ | **우도** (Likelihood) | $A$가 참일 때 $B$를 관찰할 확률 |
| $P(A)$ | **사전 확률** (Prior) | 증거 없이 $A$의 기본 확률 |
| $P(B)$ | **증거** (Evidence) | $B$가 관찰될 전체 확률 |

전체 확률의 법칙으로 분모를 전개하면:

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B \mid A) \cdot P(A) + P(B \mid A^c) \cdot P(A^c)}$$

---

## 3. 문제 1: 격자무늬 지문 문제

### 문제 상황

> 살인 현장에서 **전체 인구의 0.1%만 가진 격자무늬 지문**이 발견되었다.  
> 용의자 A도 이 지문을 보유하고 있다.  
> **용의자 A가 범인일 확률은 얼마인가?**

### 직관적 오류

많은 사람들은 "지문이 일치하니까 범인일 확률이 99.9%!"라고 생각합니다.  
이것이 바로 **기저율 무시 오류 (Base Rate Fallacy)**입니다.

In [ ]:
# === 문제 1: 격자무늬 지문 문제 ===

# 설정
N = 1_000_000  # 도시 인구 (100만)
p_fingerprint_given_guilty = 1.0   # P(지문|범인) = 1 (범인이면 반드시 지문 일치)
p_fingerprint_given_innocent = 0.001  # P(지문|무고) = 0.1%

# 사전확률: 범인은 인구 중 1명
p_guilty = 1 / N
p_innocent = 1 - p_guilty

print("=" * 60)
print(" 격자무늬 지문 문제 - 베이즈 정리 적용")
print("=" * 60)
print()
print(f"도시 인구 (N): {N:,}명")
print(f"격자무늬 지문 보유율: {p_fingerprint_given_innocent*100:.1f}%")
print()
print("--- 확률 설정 ---")
print(f"P(범인) = 1/N = {p_guilty:.6e}")
print(f"P(무고) = 1 - 1/N = {p_innocent:.10f}")
print(f"P(지문 일치 | 범인) = {p_fingerprint_given_guilty}")
print(f"P(지문 일치 | 무고) = {p_fingerprint_given_innocent}")

In [ ]:
# === 베이즈 정리 계산 ===

# P(지문 일치) = P(지문|범인)P(범인) + P(지문|무고)P(무고)
p_fingerprint = (p_fingerprint_given_guilty * p_guilty +
                 p_fingerprint_given_innocent * p_innocent)

# P(범인 | 지문 일치)
p_guilty_given_fingerprint = (p_fingerprint_given_guilty * p_guilty) / p_fingerprint

print("=" * 60)
print(" 베이즈 정리 계산")
print("=" * 60)
print()
print("P(범인|지문) = P(지문|범인) × P(범인) / P(지문)")
print()
print("[분모] P(지문 일치) = P(지문|범인)×P(범인) + P(지문|무고)×P(무고)")
print(f"  = {p_fingerprint_given_guilty} × {p_guilty:.6e} + {p_fingerprint_given_innocent} × {p_innocent:.6f}")
print(f"  = {p_fingerprint_given_guilty * p_guilty:.6e} + {p_fingerprint_given_innocent * p_innocent:.6e}")
print(f"  = {p_fingerprint:.6e}")
print()
print("[결과]")
print(f"  P(범인 | 지문 일치) = {p_guilty_given_fingerprint:.6f}")
print(f"                     ≈ {p_guilty_given_fingerprint*100:.2f}%")
print()
print(f"  ※ 직관적 답: 99.9% → 실제: {p_guilty_given_fingerprint*100:.2f}%")
print(f"  ※ 약 {int(1/p_guilty_given_fingerprint)}명 중 1명만 실제 범인!")

In [ ]:
# === 빈도 기반 직관적 설명 ===

print("=" * 60)
print(" 빈도(Frequency) 기반 직관적 이해")
print("=" * 60)
print()

n_with_fingerprint = int(N * p_fingerprint_given_innocent)
total_matches = n_with_fingerprint + 1  # 범인 1명 + 무고한 지문 보유자

print(f"도시 인구 {N:,}명 중:")
print(f"  - 실제 범인: 1명 (지문 일치)")
print(f"  - 무고하지만 같은 지문: {N:,} × 0.001 = {n_with_fingerprint:,}명")
print(f"  - 지문이 일치하는 총 인원: {total_matches:,}명")
print()
print(f"→ 용의자 A가 범인일 확률 = 1 / {total_matches:,} ≈ {1/total_matches:.4f}")
print()
print("직관적으로:")
print(f"  지문이 일치하는 {total_matches:,}명 중 범인은 단 1명!")
print(f"  나머지 {n_with_fingerprint:,}명은 무고한 시민!")

In [ ]:
# === 시각화: 확률 트리 다이어그램 ===

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# 색상 정의
c_guilty = '#e74c3c'
c_innocent = '#3498db'
c_match = '#f39c12'
c_no_match = '#95a5a6'

# --- 레벨 1: 범인 vs 무고 ---
# 시작점
ax.annotate('전체 인구\n1,000,000명', xy=(1, 5), fontsize=11, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#ecf0f1', edgecolor='black'))

# 범인 가지
ax.annotate('', xy=(3.5, 7.5), xytext=(2, 5.5),
            arrowprops=dict(arrowstyle='->', color=c_guilty, lw=2))
ax.text(2.2, 6.8, 'P(범인)\n= 1/1,000,000', fontsize=8, color=c_guilty, ha='center')
ax.annotate('범인\n1명', xy=(4, 7.5), fontsize=10, fontweight='bold',
            ha='center', va='center', color='white',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=c_guilty))

# 무고 가지
ax.annotate('', xy=(3.5, 2.5), xytext=(2, 4.5),
            arrowprops=dict(arrowstyle='->', color=c_innocent, lw=2))
ax.text(2.2, 3.2, 'P(무고)\n≈ 1', fontsize=8, color=c_innocent, ha='center')
ax.annotate('무고\n999,999명', xy=(4, 2.5), fontsize=10, fontweight='bold',
            ha='center', va='center', color='white',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=c_innocent))

# --- 레벨 2: 지문 일치 ---
# 범인 → 지문 일치
ax.annotate('', xy=(6.5, 8.5), xytext=(5, 7.8),
            arrowprops=dict(arrowstyle='->', color=c_match, lw=2))
ax.text(5.5, 8.5, 'P(지문|범인)\n= 1.0', fontsize=8, color=c_match, ha='center')
ax.annotate('지문 일치\n1명', xy=(7.5, 8.5), fontsize=9, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#fdebd0', edgecolor=c_match))

# 범인 → 지문 불일치 (불가능)
ax.annotate('', xy=(6.5, 6.5), xytext=(5, 7.2),
            arrowprops=dict(arrowstyle='->', color=c_no_match, lw=1.5))
ax.text(5.5, 6.5, 'P(불일치|범인)\n= 0', fontsize=8, color=c_no_match, ha='center')
ax.annotate('불일치\n0명', xy=(7.5, 6.5), fontsize=9,
            ha='center', va='center', color='gray',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f5', edgecolor='gray'))

# 무고 → 지문 일치
ax.annotate('', xy=(6.5, 3.5), xytext=(5, 2.8),
            arrowprops=dict(arrowstyle='->', color=c_match, lw=2))
ax.text(5.5, 3.5, 'P(지문|무고)\n= 0.001', fontsize=8, color=c_match, ha='center')
ax.annotate('지문 일치\n1,000명', xy=(7.5, 3.5), fontsize=9, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#fdebd0', edgecolor=c_match))

# 무고 → 지문 불일치
ax.annotate('', xy=(6.5, 1.5), xytext=(5, 2.2),
            arrowprops=dict(arrowstyle='->', color=c_no_match, lw=1.5))
ax.text(5.5, 1.5, 'P(불일치|무고)\n= 0.999', fontsize=8, color=c_no_match, ha='center')
ax.annotate('불일치\n998,999명', xy=(7.5, 1.5), fontsize=9,
            ha='center', va='center', color='gray',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f5', edgecolor='gray'))

# --- 최종 결과 ---
ax.annotate('지문 일치 총 1,001명 중\n실제 범인 = 1명\n\nP(범인|지문) = 1/1,001\n≈ 0.001 (0.1%)',
            xy=(9.2, 6), fontsize=11, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#fadbd8',
                      edgecolor=c_guilty, linewidth=2))

ax.set_title('확률 트리: 격자무늬 지문 문제 (도시 인구 100만)', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# === 감도 분석: 도시 인구 크기에 따른 확률 변화 ===

populations = np.logspace(2, 7, 200)  # 100 ~ 10,000,000
p_guilty_by_pop = []

for pop in populations:
    prior = 1 / pop
    evidence = (1.0 * prior) + (0.001 * (1 - prior))
    posterior = (1.0 * prior) / evidence
    p_guilty_by_pop.append(posterior)

fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogx(populations, p_guilty_by_pop, color='#e74c3c', linewidth=2.5)

# 주요 포인트 표시
key_pops = [1000, 10000, 100000, 1000000]
for kp in key_pops:
    prior_k = 1 / kp
    ev_k = (1.0 * prior_k) + (0.001 * (1 - prior_k))
    post_k = (1.0 * prior_k) / ev_k
    ax.plot(kp, post_k, 'o', color='#2c3e50', markersize=8, zorder=5)
    ax.annotate(f'N={kp:,}\nP={post_k:.3f}',
                (kp, post_k), textcoords='offset points',
                xytext=(15, 10), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='gray'),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='P = 0.5 (반반)')
ax.set_xlabel('도시 인구 (N)', fontsize=12)
ax.set_ylabel('P(범인 | 지문 일치)', fontsize=12)
ax.set_title('도시 인구 크기에 따른 P(범인|지문 일치) 변화', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("\n핵심 통찰:")
print("  - 도시 인구가 클수록 지문 일치만으로는 범인을 특정할 수 없다")
print("  - 인구 1,000명: P(범인) ≈ 50%, 인구 100만명: P(범인) ≈ 0.1%")
print("  - 이것이 법정에서 지문 증거만으로 유죄를 확정하면 안 되는 이유!")

### 해석 - 문제 1

**왜 직관이 틀리는가?**

1. **기저율(Base Rate)을 무시**: 범인은 100만 명 중 단 1명이라는 사실을 간과
2. **우도(Likelihood)에만 집중**: "지문이 일치한다" = 유죄로 바로 연결
3. **실제로는**: 지문이 일치하는 1,001명 중 범인은 단 1명

**법정에서의 시사점**:
- DNA 증거, 지문 증거 등은 강력하지만 **유일한 증거**가 될 수 없음
- 추가 증거(알리바이, 동기, 목격자 등)가 반드시 필요
- 이를 무시하는 오류를 **검찰관의 오류(Prosecutor's Fallacy)**라 함

---

## 4. 문제 2: 택시 문제 (Cab Problem)

### 문제 상황

> 어느 도시에서 택시의 **85%는 녹색**, **15%는 청색**이다.  
> 밤에 사고가 발생했고, 목격자가 "**청색 택시였다**"고 증언했다.  
> 목격자의 색상 판별 정확도는 **80%**이다.  
> **사고 택시가 실제로 청색일 확률은?**

### 직관적 오류

대부분의 사람들은 **80%**라고 답합니다.  
"목격자가 80% 정확하니까 실제로 청색일 확률도 80%"  
하지만 실제 답은 **약 41%**입니다!

In [ ]:
# === 문제 2: 택시 문제 ===

# 설정
p_blue = 0.15       # P(청색) - 사전확률
p_green = 0.85      # P(녹색)
p_say_blue_given_blue = 0.80   # P(청색 증언|실제 청색) - 정확한 판별
p_say_blue_given_green = 0.20  # P(청색 증언|실제 녹색) - 오판

print("=" * 60)
print(" 택시 문제 - 베이즈 정리 적용")
print("=" * 60)
print()
print("[주어진 정보]")
print(f"  P(청색 택시) = {p_blue} (15%)")
print(f"  P(녹색 택시) = {p_green} (85%)")
print(f"  P(청색이라 증언 | 실제 청색) = {p_say_blue_given_blue} (80%)")
print(f"  P(청색이라 증언 | 실제 녹색) = {p_say_blue_given_green} (20%)")
print()

# P(청색 증언) = P(청색증언|청색)P(청색) + P(청색증언|녹색)P(녹색)
p_say_blue = (p_say_blue_given_blue * p_blue) + (p_say_blue_given_green * p_green)

# P(실제 청색 | 청색 증언)
p_blue_given_say_blue = (p_say_blue_given_blue * p_blue) / p_say_blue

print("[베이즈 정리 계산]")
print()
print("P(실제 청색 | 청색 증언)")
print(f"  = P(청색 증언|청색) × P(청색) / P(청색 증언)")
print()
print(f"분자: P(청색 증언|청색) × P(청색)")
print(f"    = {p_say_blue_given_blue} × {p_blue}")
print(f"    = {p_say_blue_given_blue * p_blue:.4f}")
print()
print(f"분모: P(청색 증언)")
print(f"    = P(청색 증언|청색)×P(청색) + P(청색 증언|녹색)×P(녹색)")
print(f"    = {p_say_blue_given_blue}×{p_blue} + {p_say_blue_given_green}×{p_green}")
print(f"    = {p_say_blue_given_blue * p_blue:.4f} + {p_say_blue_given_green * p_green:.4f}")
print(f"    = {p_say_blue:.4f}")
print()
print(f"결과: P(실제 청색 | 청색 증언) = {p_say_blue_given_blue * p_blue:.4f} / {p_say_blue:.4f}")
print(f"                             = {p_blue_given_say_blue:.4f}")
print(f"                             ≈ {p_blue_given_say_blue*100:.1f}%")
print()
print("=" * 60)
print(f"  직관적 답: 80%")
print(f"  정확한 답: {p_blue_given_say_blue*100:.1f}%")
print("=" * 60)

In [ ]:
# === 빈도 기반 설명: Contingency Table ===

# 택시 100대로 계산
total_cabs = 100

blue_cabs = int(total_cabs * p_blue)  # 15대
green_cabs = int(total_cabs * p_green)  # 85대

# 목격자 증언 결과
blue_say_blue = blue_cabs * p_say_blue_given_blue  # 청색인데 청색이라 함: 12대
blue_say_green = blue_cabs * (1 - p_say_blue_given_blue)  # 청색인데 녹색이라 함: 3대
green_say_blue = green_cabs * p_say_blue_given_green  # 녹색인데 청색이라 함: 17대
green_say_green = green_cabs * (1 - p_say_blue_given_green)  # 녹색인데 녹색이라 함: 68대

# Contingency Table 생성
data = {
    '실제 청색': [blue_say_blue, blue_say_green, blue_cabs],
    '실제 녹색': [green_say_blue, green_say_green, green_cabs],
    '합계': [blue_say_blue + green_say_blue,
            blue_say_green + green_say_green,
            total_cabs]
}

df_contingency = pd.DataFrame(
    data,
    index=['"청색" 증언', '"녹색" 증언', '합계']
)

print("Contingency Table (택시 100대 기준)")
print("=" * 50)
print(df_contingency.to_string())
print()
print(f"\n'청색' 증언 총 {blue_say_blue + green_say_blue:.0f}건 중:")
print(f"  - 실제 청색: {blue_say_blue:.0f}건")
print(f"  - 실제 녹색(오판): {green_say_blue:.0f}건")
print(f"  → P(실제 청색|청색 증언) = {blue_say_blue:.0f}/{blue_say_blue+green_say_blue:.0f} ≈ {p_blue_given_say_blue:.2%}")

In [ ]:
# === 시각화: Contingency Table 히트맵 + 확률 트리 ===

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- 왼쪽: Contingency Table 히트맵 ---
ax1 = axes[0]
heatmap_data = np.array([
    [blue_say_blue, green_say_blue],
    [blue_say_green, green_say_green]
])

sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=['실제 청색 (15대)', '실제 녹색 (85대)'],
            yticklabels=['"청색" 증언', '"녹색" 증언'],
            ax=ax1, linewidths=2, linecolor='white',
            annot_kws={'size': 16, 'fontweight': 'bold'},
            cbar_kws={'label': '택시 수'})

ax1.set_title('Contingency Table (택시 100대)', fontsize=13, fontweight='bold')

# 핵심 강조
ax1.text(0, -0.3, f'"청색" 증언 {blue_say_blue+green_say_blue:.0f}대 중 실제 청색은 {blue_say_blue:.0f}대!',
         fontsize=11, fontweight='bold', color='#c0392b',
         transform=ax1.transAxes)

# --- 오른쪽: 확률 트리 ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')

# 시작
ax2.text(1, 5, '택시\n사고', fontsize=11, fontweight='bold', ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#ecf0f1', edgecolor='black'))

# 청색/녹색 분기
ax2.annotate('', xy=(3.5, 7.5), xytext=(2, 5.5),
             arrowprops=dict(arrowstyle='->', color='#2980b9', lw=2))
ax2.text(2.2, 6.8, '15%', fontsize=10, color='#2980b9', fontweight='bold')
ax2.text(4, 7.5, '청색', fontsize=11, fontweight='bold', ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#aed6f1'))

ax2.annotate('', xy=(3.5, 2.5), xytext=(2, 4.5),
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax2.text(2.2, 3.2, '85%', fontsize=10, color='#27ae60', fontweight='bold')
ax2.text(4, 2.5, '녹색', fontsize=11, fontweight='bold', ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#abebc6'))

# 증언 분기
ax2.annotate('', xy=(6.5, 8.5), xytext=(5, 7.8),
             arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))
ax2.text(5.5, 8.6, '80%', fontsize=9, color='#e74c3c')
ax2.text(7.2, 8.5, '"청색"\n0.15×0.80\n=0.12', fontsize=9, ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fadbd8'))

ax2.annotate('', xy=(6.5, 6.5), xytext=(5, 7.2),
             arrowprops=dict(arrowstyle='->', color='gray', lw=1))
ax2.text(5.5, 6.6, '20%', fontsize=9, color='gray')
ax2.text(7.2, 6.5, '"녹색"\n0.15×0.20\n=0.03', fontsize=9, ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f5'))

ax2.annotate('', xy=(6.5, 3.5), xytext=(5, 2.8),
             arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))
ax2.text(5.5, 3.6, '20%', fontsize=9, color='#e74c3c')
ax2.text(7.2, 3.5, '"청색"\n0.85×0.20\n=0.17', fontsize=9, ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fadbd8'))

ax2.annotate('', xy=(6.5, 1.5), xytext=(5, 2.2),
             arrowprops=dict(arrowstyle='->', color='gray', lw=1))
ax2.text(5.5, 1.6, '80%', fontsize=9, color='gray')
ax2.text(7.2, 1.5, '"녹색"\n0.85×0.80\n=0.68', fontsize=9, ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f5'))

# 결과
ax2.text(9.3, 5.5, 'P(청색|"청색")\n= 0.12/(0.12+0.17)\n= 0.12/0.29\n≈ 41.4%',
         fontsize=11, fontweight='bold', ha='center', va='center',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fadbd8',
                   edgecolor='#e74c3c', linewidth=2))

ax2.set_title('확률 트리: 택시 문제', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# === 감도 분석: 목격자 정확도에 따른 확률 변화 ===

accuracies = np.linspace(0.5, 1.0, 200)
posterior_blue = []

for acc in accuracies:
    p_sb_b = acc              # P(청색 증언|실제 청색)
    p_sb_g = 1 - acc          # P(청색 증언|실제 녹색)
    p_sb = p_sb_b * p_blue + p_sb_g * p_green
    post = (p_sb_b * p_blue) / p_sb
    posterior_blue.append(post)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(accuracies * 100, np.array(posterior_blue) * 100,
        color='#2980b9', linewidth=2.5, label='P(실제 청색 | 청색 증언)')

# 기준선
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='P = 50%')
ax.axhline(y=15, color='#27ae60', linestyle=':', alpha=0.7, label='기저율: 15% (청색 비율)')

# 80% 정확도 포인트
ax.plot(80, p_blue_given_say_blue * 100, 'o', color='#e74c3c', markersize=12, zorder=5)
ax.annotate(f'정확도 80% → P = {p_blue_given_say_blue*100:.1f}%\n(직관: 80%와 큰 차이!)',
            xy=(80, p_blue_given_say_blue * 100),
            xytext=(58, 55), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#fadbd8', alpha=0.9))

# 50% 넘기려면 몇 %?
for i, acc in enumerate(accuracies):
    if posterior_blue[i] >= 0.5:
        threshold_acc = acc
        break

ax.axvline(x=threshold_acc * 100, color='#f39c12', linestyle='--', alpha=0.7)
ax.annotate(f'P > 50%가 되려면\n정확도 > {threshold_acc*100:.0f}% 필요',
            xy=(threshold_acc * 100, 50),
            xytext=(threshold_acc * 100 + 5, 30), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#f39c12'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# 대각선 (직관적 오류: 정확도 = 사후확률)
ax.plot([50, 100], [50, 100], '--', color='#e74c3c', alpha=0.3, linewidth=1.5,
        label='직관적 오류선 (정확도 = 사후확률)')

ax.set_xlabel('목격자 판별 정확도 (%)', fontsize=12)
ax.set_ylabel('P(실제 청색 | 청색 증언) (%)', fontsize=12)
ax.set_title('목격자 정확도에 따른 사후 확률 변화', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(50, 100)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n사후확률이 50%를 넘으려면 목격자 정확도가 {threshold_acc*100:.0f}% 이상이어야 함!")
print("기저율이 낮은(15%) 경우, 80% 정확도로도 '반반'조차 되지 않는다.")

### 해석 - 문제 2

**왜 80%가 아니라 41%인가?**

| 택시 100대 | "청색" 증언 발생 |
|---|---|
| 청색 15대 중 → 12대 정확히 "청색" 증언 | 12대 |
| 녹색 85대 중 → 17대 잘못 "청색" 증언 | 17대 |
| **합계** | **29대** |

"청색" 증언이 나온 29대 중 실제 청색은 12대 = **41.4%**

**핵심 원인**: 녹색 택시가 압도적으로 많기 때문에(85%), 20%의 오판율도 절대적 숫자로는 큰 영향을 미침

| 직관적 추론 | 베이지안 추론 |
|---|---|
| 정확도 80% → 확률 80% | 기저율 + 정확도 → 확률 41% |
| 기저율 무시 | 기저율 반영 |
| 우도만 고려 | 사전확률 × 우도 ÷ 증거 |

---

## 5. 기저율 무시 오류 (Base Rate Fallacy) 종합

In [ ]:
# === 두 문제 비교 요약 ===

comparison = pd.DataFrame({
    '항목': ['사건', '기저율 (사전확률)', '증거의 강도 (우도)',
            '직관적 답', '정확한 답 (베이즈)', '오류 원인'],
    '지문 문제': [
        '지문 일치 → 범인?',
        'P(범인) = 1/1,000,000',
        'P(지문|범인) = 1.0',
        '~99.9%',
        '~0.1%',
        '극히 낮은 기저율 무시'
    ],
    '택시 문제': [
        '목격 증언 → 청색?',
        'P(청색) = 15%',
        'P(청색증언|청색) = 80%',
        '~80%',
        '~41%',
        '낮은 기저율 무시'
    ]
})

print(comparison.to_string(index=False))

## 6. ML 연관: 불균형 데이터에서 정밀도(Precision)의 함정

기저율 무시 오류는 머신러닝에서도 자주 발생합니다.

In [ ]:
# === ML 연관: 불균형 데이터의 Precision 함정 ===

print("=" * 65)
print(" ML에서의 기저율 무시: 스팸 필터 예시")
print("=" * 65)
print()
print("상황: 이메일의 1%만 스팸 (기저율 = 1%)")
print("모델: 스팸 탐지 정확도 99%")
print()

# 설정
p_spam = 0.01
p_not_spam = 0.99
sensitivity = 0.99      # P(스팸이라 예측|실제 스팸) = 99%
false_positive = 0.01   # P(스팸이라 예측|정상) = 1%

# 이메일 10,000통
total_emails = 10000
actual_spam = total_emails * p_spam           # 100통
actual_normal = total_emails * p_not_spam     # 9,900통

detected_spam_correct = actual_spam * sensitivity        # 99통 (정확)
detected_spam_wrong = actual_normal * false_positive     # 99통 (오탐!)
total_detected = detected_spam_correct + detected_spam_wrong

precision = detected_spam_correct / total_detected

print(f"이메일 {total_emails:,}통:")
print(f"  실제 스팸: {int(actual_spam)}통")
print(f"  정상 메일: {int(actual_normal):,}통")
print()
print(f"모델이 '스팸'이라 판단한 메일:")
print(f"  진짜 스팸: {int(detected_spam_correct)}통 (True Positive)")
print(f"  정상인데 스팸 판정: {int(detected_spam_wrong)}통 (False Positive!)")
print(f"  총 '스팸' 판정: {int(total_detected)}통")
print()
print(f"Precision = {int(detected_spam_correct)} / {int(total_detected)} = {precision:.1%}")
print()
print("결론: 99% 정확한 모델도, 기저율이 낮으면 Precision은 50%에 불과!")
print("     '스팸'이라 판정된 메일 중 절반은 정상 메일!")
print()
print("교훈: 불균형 데이터에서는 Accuracy보다 Precision, Recall, F1을 봐야 한다!")

In [ ]:
# === 최종 요약 ===

print("=" * 65)
print(" A13. 베이즈 정리 응용 최종 요약")
print("=" * 65)
print()
print("[베이즈 정리]")
print("  P(A|B) = P(B|A) x P(A) / P(B)")
print()
print("[문제 1: 지문 문제]")
print("  인구 100만, 지문 보유율 0.1% → P(범인|지문) ≈ 0.1%")
print("  직관적 오류: 지문 일치 = 범인 확실 (99.9%)")
print()
print("[문제 2: 택시 문제]")
print("  택시 85% 녹색/15% 청색, 목격 정확도 80%")
print("  P(실제 청색|청색 증언) ≈ 41% (직관: 80%)")
print()
print("[핵심 교훈]")
print("  1. 기저율(Base Rate)을 절대 무시하지 말 것")
print("  2. 증거의 강도(우도)만으로 결론 내리면 위험")
print("  3. 베이즈 정리는 직관의 함정을 수학적으로 교정")
print("  4. ML에서도 불균형 데이터 → Precision 함정 주의")
print("=" * 65)